In [21]:
from tile_mate.stitcher import get_all_tile_data
from utils import vectorize_all_tiles
import geopandas as gpd

In [8]:
%%time

df_umd_ocean = get_all_tile_data('umd_ocean_mask')
ocean_urls = df_umd_ocean.url.tolist()

CPU times: user 137 μs, sys: 157 μs, total: 294 μs
Wall time: 313 μs


In [9]:
output_parquet = 'umd_land_mask.parquet'
df_land = vectorize_all_tiles(ocean_urls, output_parquet, n_workers=5)

100%|██████████████████████████| 540/540 [42:06<00:00,  4.68s/it]


Individual tile processing complete. Combining results...


In [10]:
df_land.to_parquet('land_mask.parquet', compression='zstd')

In [11]:
df_land.to_file('land_mask.geojson', driver='GeoJSON')

# Let's merge contigiuous areas.

In [19]:
%%time

union_geo = df_land.geometry.union_all()

In [23]:
df_land_union = gpd.GeoDataFrame(geometry=[union_geo], crs=df_land.crs)

In [28]:
%%time

df_land_union_e = df_land_union.explode().reset_index(drop=True)

CPU times: user 188 ms, sys: 333 ms, total: 520 ms
Wall time: 518 ms


In [31]:
df_land_union_e.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- undefined
Datum: World Geodetic System 1984
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [29]:
df_land_union_e.head()

,geometry
0,"POLYGON ((-160.77875 -59.99975, -160.77875 -59..."
1,"POLYGON ((-160.78075 -59.99975, -160.78075 -59..."
2,"POLYGON ((-160.78575 -59.99975, -160.78575 -59..."
3,"POLYGON ((-160.79775 -59.99975, -160.79775 -59..."
4,"POLYGON ((-160.77175 -59.99975, -160.77175 -59..."


In [ ]:
df_land_union_e.to_parquet('land_mask_umd_union.parquet', compression='zstd')

In [36]:
geo = df_land_union_e.union_all()

KeyboardInterrupt: 